# T08 — Encoder, decoder, complejidad y qué NO dice el título

## 1. Título y paper

**Paper:** *Attention Is All You Need* (Vaswani et al., 2017)  
**Fuente primaria:** [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)  
**Foco de esta miniatura:** el modelo completo y su lectura honesta  
**Ficha completa:** [`P08_transformer`](../../papers/foundational/P08_transformer/README.md)


## 2. Objetivos

1. Describir el flujo encoder → cross-attention → decoder.
2. Enunciar con precisión los límites del paper y del título.


## 3. Prerrequisitos

- Python 3.11+ con el paquete instalado (`pip install -e .`).
- Notebook [`P08_transformer`](P08_transformer.ipynb) al menos hojeado.
- Álgebra de vectores: producto escalar, norma y softmax.


## 4. Intuición

El encoder lee toda la frase de origen sin restricciones. El decoder escribe de izquierda a derecha, mirando lo ya escrito (self-attention causal) y lo que el encoder entendió (cross-attention).


## 5. Concepto mínimo

```text
encoder ×N : self-attention → FFN                (sin máscara)
decoder ×N : self-attention causal → cross-attention → FFN
```

Modelo base del paper: N=6, d_model=512, h=8, d_ff=2048. La familia BERT usa solo el encoder; la familia GPT, solo el decoder.


## 6. Código explicado

Código mínimo, sin dependencias externas.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('transformer', seed=7)['result']
show(r['complejidad'])
print('normas tras residual + layer norm:', r['norma_tras_layernorm'])

## 7. Predicción antes de ejecutar

¿Qué crece más rápido al multiplicar n por 10: el coste de la atención o el del bloque recurrente?

> Escribe tu respuesta antes de continuar.


## 8. Experimento controlado


In [ ]:
memoria_bytes = lambda n: n * n * 4          # matriz de atención en float32
for n in (512, 2048, 8192, 128000):
    mb = memoria_bytes(n) / 1024 ** 2
    print(f'n={n:>7} → matriz de atención ≈ {mb:>12,.1f} MB por cabeza y capa')

## 9. Salida interpretable

Con n=128 000 la matriz de atención de UNA cabeza y UNA capa ocupa decenas de gigabytes. Por eso el contexto largo real no usa atención densa ingenua: usa variantes (atención dispersa, kernels de E/S optimizada, compresión). Ese es trabajo **posterior** al paper.


## 10. Comentario pedagógico

Esta miniatura aísla **una** pieza del bloque. Aislar es didáctico y también es una simplificación: en el modelo real todas las piezas interactúan y se entrenan juntas.


## 11. Error o anti-patrón deliberado


In [ ]:
print('«Attention Is All You Need» leído literalmente diría que basta la atención.')
print('El propio modelo del paper necesita, además:')
for pieza in ['FFN por posición', 'conexiones residuales', 'layer normalization',
              'codificación posicional', 'embeddings compartidos', 'label smoothing', 'warmup del LR']:
    print('  -', pieza)

## 12. Corrección


In [ ]:
lectura_correcta = {
    'que_elimina': ['recurrencia', 'convolución'],
    'que_conserva': ['FFN', 'residual', 'layer norm', 'embeddings', 'codificación posicional'],
    'que_gana': 'paralelización y camino O(1) entre posiciones',
    'que_paga': 'coste y memoria O(n²) en la longitud de secuencia',
    'que_NO_dice': 'que la atención sola baste para construir un modelo entrenable',
}
show(lectura_correcta)

## 13. Desafío guiado

Escribe en cinco líneas qué hereda BERT del encoder y qué hereda GPT del decoder, sin usar la palabra «Transformer».


## 14. Desafío autónomo

Reescribe esta pieza con proyecciones aprendidas y comprueba que tu implementación reproduce las propiedades verificadas aquí (sumas, formas, invariantes). Documenta la semilla.


## 15. Evidencia de aprendizaje

Guarda la salida del experimento, tu predicción previa y una frase sobre qué invariante acabas de verificar.


## 16. Cierre

Pieza cubierta: **el modelo completo y su lectura honesta**. Ya puede describirse con precisión, sin metáforas.


## 17. Conexión con el siguiente hito

Con el bloque desmontado, las dos ramas —encoder (P09) y decoder (P10)— se leen sin misterio.
